# 10 dpa to 14 dpa 3D Model Alignment

Reconstruct both stages, calculate baseline morphology, align them with the current Spateo API, and save the aligned AnnData objects.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import numpy as np
import spateo as st

# Enable transparent backgrounds.
import pyvista as pv

pv.global_theme.transparent_background = True

import warnings

warnings.filterwarnings("ignore")


In [ ]:
cpo_10_xy1 = [
    (1305.8485686301365, 1893.0791706372074, -4490.316748863963),
    (944.0, 1655.0, 75.0),
    (0.9772773187740892, -0.2011065155392658, 0.06697172252064593),
]
cpo_10_xy2 = [
    (861.4007529132589, 2028.3753467911467, 4644.846967936437),
    (944.0, 1655.0, 75.0),
    (-0.9817719930373955, 0.18716905163369624, -0.03303785401510178),
]
cpo_10_z = [
    (-3469.0966239329905, 2901.617397725647, 58.98915719650984),
    (944.0, 1655.0, 75.0),
    (0.005184768895382603, 0.005511315408809347, -0.9999713713771842),
]
cpo_14_xy1 = [
    (1413.0038531541918, 1599.9930119308501, -4486.440859263499),
    (944.0, 1655.0, 75.0),
    (0.9931843627703779, -0.05496213373763433, 0.10278125025219716),
]
cpo_14_xy2 = [
    (8.826402963346666, 1502.8440395783352, 4561.873299156306),
    (944.0, 1655.0, 75.0),
    (-0.9760966205121214, 0.08362042508532216, -0.20060660989450432),
]
cpo_14_z = [
    (-4582.861825366593, 1300.1039027987563, 417.7526057063721),
    (944.0, 1655.0, 75.0),
    (0.06364890563804504, -0.027764419618153784, 0.9975860633621274),
]


## Load and validate data


In [ ]:
stage1_adata = st.read_h5ad("/DATA/User/gaomohan/Planarian_project/data/single_cell/10dpa1.sc.h5ad")
stage2_adata = st.read_h5ad("/DATA/User/gaomohan/Planarian_project/data/single_cell/14dpa1.sc.h5ad")
stage1_adata, stage2_adata


### Validate the alignment input contract


In [ ]:
for stage_name, adata in {"10 dpa": stage1_adata, "14 dpa": stage2_adata}.items():
    if "anno" not in adata.obs:
        raise KeyError(f"{stage_name} is missing obs['anno'].")
    if "spatial_3d" not in adata.obsm:
        raise KeyError(f"{stage_name} is missing obsm['spatial_3d'].")
    coordinates = np.asarray(adata.obsm["spatial_3d"])
    if coordinates.shape != (adata.n_obs, 3) or not np.isfinite(coordinates).all():
        raise ValueError(f"{stage_name} requires finite (n_obs, 3) spatial coordinates.")
    if not adata.obs_names.is_unique:
        raise ValueError(f"{stage_name} has duplicated observation identifiers.")
    print(stage_name, adata.shape, adata.obs["anno"].value_counts(dropna=False).to_dict())


In [ ]:
stage1_adata.obs["anno"], stage2_adata.obs["anno"]


In [ ]:
for stage_name, adata in {"stage 1": stage1_adata, "stage 2": stage2_adata}.items():
    if "counts_X" not in adata.layers:
        warnings.warn(
            f"{stage_name} has no count layer; treating X as counts. Verify this assumption."
        )
        adata.layers["counts_X"] = adata.X.copy()
    st.pp.normalize_total(
        adata,
        layer="counts_X",
        out_layer="norm_X",
        target_sum=None,
        size_factor_key="Size_Factor",
        inplace=True,
    )
    st.pp.log1p_layer(
        adata,
        layer="norm_X",
        out_layer="log1p_X",
        set_X=False,
        inplace=True,
    )


## Construct the point-cloud model


In [ ]:
stage1_pc, plot_cmap = st.tdr.construct_pc(
    adata=stage1_adata.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

st.pl.three_d_plot(
    model=stage1_pc,
    key="tissue",
    model_style="points",
    show_axes=True,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(400, 400),
    cpo=cpo_10_xy1,
    # filename = f"./figures/10dpa.pdf"
)


## Reconstruct the surface mesh


In [ ]:
stage1_mesh, _, _ = st.tdr.construct_surface(
    pc=stage1_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.10},
    smooth=5000,
    scale_factor=1.08,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_mesh, stage1_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    # opacity=[0.6, 1],
    window_size=(400, 400),
    cpo=cpo_10_z,
    # filename = f"./figures_2/10dpa_pc_mesh_z.pdf",
)


In [ ]:
stage2_pc, plot_cmap = st.tdr.construct_pc(
    adata=stage2_adata.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)


In [ ]:
st.pl.three_d_plot(
    model=stage2_pc,
    key="tissue",
    model_style="points",
    show_axes=True,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(400, 400),
    cpo=cpo_14_xy2,
    # filename = f"./figures_2/14dpa.pdf"
)


In [ ]:
stage2_mesh, _, _ = st.tdr.construct_surface(
    pc=stage2_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.08},
    smooth=5000,
    scale_factor=1.08,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([stage2_mesh, stage2_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    # opacity=[0.6, 1],
    window_size=(400, 400),
    cpo=cpo_14_z,
    # filename = f"./figures_2/14_dpa_pc_mesh_z.pdf",
)


In [ ]:
um_stage1_pc_model, um_stage1_mesh_model = stage1_pc.copy(), stage1_mesh.copy()

um_stage1_pc_model.points = um_stage1_pc_model.points / 1000
um_stage1_mesh_model.points = um_stage1_mesh_model.points / 1000


## Calculate morphological features


In [ ]:
morph = st.tdr.model_morphology(model=um_stage1_mesh_model, pc=um_stage1_pc_model)
morph


In [ ]:
um_stage2_pc_model, um_stage2_mesh_model = stage2_pc.copy(), stage2_mesh.copy()

um_stage2_pc_model.points = um_stage2_pc_model.points / 1000
um_stage2_mesh_model.points = um_stage2_mesh_model.points / 1000


In [ ]:
morph = st.tdr.model_morphology(model=um_stage2_mesh_model, pc=um_stage2_pc_model)
morph


In [ ]:
stage1_adata.obsm["spatial_3d"] = stage1_adata.obsm["spatial_3d"].astype(np.float64)

stage2_adata.obsm["spatial_3d"] = stage2_adata.obsm["spatial_3d"].astype(np.float64)


## Align developmental stages


In [ ]:
align_samples, align_samples_ref, _, _ = st.align.morpho_align_ref(
    models=[stage1_adata, stage2_adata],
    models_ref=None,
    n_sampling=10000,
    sampling_method="trn",
    rep_layer="log1p_X",
    rep_field="layer",
    spatial_key="spatial_3d",
    key_added="3d_align_spatial",
    device="0",
)
align_samples


In [ ]:
stage1_aligned = align_samples[0].copy()
stage2_aligned = align_samples[1].copy()


In [ ]:
stage1_raw_pc, _ = st.tdr.construct_pc(
    adata=stage1_aligned.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

stage1_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage1_aligned.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)


In [ ]:
stage2_raw_pc, _ = st.tdr.construct_pc(
    adata=stage2_aligned.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

stage2_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage2_aligned.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)


In [ ]:
cpo = [
    (746.583638250998, 1426.5293241132986, 4650.867243041787),
    (944.0, 1655.0, 75.0),
    (-0.9837490807936163, 0.17637001290509832, -0.033635763490226143),
]


In [ ]:
raw_pair3 = st.tdr.collect_models([stage1_aligned_pc.copy(), stage2_aligned_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair3,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    opacity=[0.6, 0.6],
    show_legend=False,
    show_axes=True,
    jupyter="static",
    window_size=(400, 400),
    cpo=cpo
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


In [ ]:
stage1_aligned.write_h5ad(
    "/DATA/User/gaomohan/Planarian_project/data/aligned/10dpa_tdr_aligned.h5ad", compression="gzip"
)


In [ ]:
stage2_aligned.write_h5ad(
    "/DATA/User/gaomohan/Planarian_project/data/aligned/14dpa_tdr_aligned.h5ad", compression="gzip"
)
